# Class Distribution Analysis for MIMIC OBS and India InternVL

This notebook loads the gold-label pickle files for both datasets, validates the schema, and shows class distributions as tables and bar charts.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [ ]:
REQUIRED_FILES = [
    Path('data/mimic_obs/annotated_merged_gold_upd2.pkl'),
    Path('data/india/internvl_gold_labels.pkl'),
]


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if all((candidate / rel_path).exists() for rel_path in REQUIRED_FILES):
            return candidate
    raise FileNotFoundError('Could not locate the repo root containing both dataset pickle files.')


repo_root = find_repo_root(Path.cwd().resolve())
dataset_paths = {
    'MIMIC OBS': repo_root / 'data/mimic_obs/annotated_merged_gold_upd2.pkl',
    'India InternVL': repo_root / 'data/india/internvl_gold_labels.pkl',
}

datasets = {name: pd.read_pickle(path) for name, path in dataset_paths.items()}
schema_summary = pd.DataFrame(
    [
        {
            'dataset': name,
            'rows': len(df),
            'columns': ', '.join(df.columns.astype(str)),
            'null_labels': int(df['label'].isna().sum()),
        }
        for name, df in datasets.items()
    ]
)
display(schema_summary)

In [ ]:
preferred_order = ['early', 'late', 'unrelated']
all_labels = set()
for df in datasets.values():
    all_labels.update(df['label'].dropna().astype(str).str.strip().unique())
label_order = [label for label in preferred_order if label in all_labels] + sorted(all_labels - set(preferred_order))

summary_frames = []
for dataset_name, df in datasets.items():
    cleaned = df.copy()
    cleaned['label'] = cleaned['label'].astype(str).str.strip()
    counts = cleaned['label'].value_counts().reindex(label_order, fill_value=0)
    summary = pd.DataFrame({
        'dataset': dataset_name,
        'label': counts.index,
        'count': counts.values,
    })
    summary['percent'] = (summary['count'] / len(cleaned) * 100).round(2)
    summary['total_rows'] = len(cleaned)
    summary_frames.append(summary)

summary_df = pd.concat(summary_frames, ignore_index=True)
summary_df

In [ ]:
count_table = summary_df.pivot(index='label', columns='dataset', values='count').reindex(label_order)
percent_table = summary_df.pivot(index='label', columns='dataset', values='percent').reindex(label_order)

print('Count table')
display(count_table)
print('Percent table')
display(percent_table)

In [ ]:
colors = {'early': '#4C78A8', 'late': '#F58518', 'unrelated': '#54A24B'}
bar_colors = [colors.get(label, '#777777') for label in label_order]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

count_table.plot(kind='bar', ax=axes[0], color=['#4C78A8', '#E45756'])
axes[0].set_title('Class Counts by Dataset')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

percent_table.plot(kind='bar', ax=axes[1], color=['#4C78A8', '#E45756'])
axes[1].set_title('Class Percentages by Dataset')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Percent')
axes[1].tick_params(axis='x', rotation=0)

plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(14, 5), constrained_layout=True)
if len(datasets) == 1:
    axes = [axes]

for ax, dataset_name in zip(axes, datasets):
    subset = summary_df[summary_df['dataset'] == dataset_name].set_index('label').reindex(label_order)
    bars = ax.bar(subset.index, subset['count'], color=bar_colors)
    ax.set_title(dataset_name)
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)
    for bar, pct in zip(bars, subset['percent']):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f'{pct:.2f}%',
            ha='center',
            va='bottom',
            fontsize=10,
        )

plt.show()

In [ ]:
percent_comparison = percent_table.copy()
percent_comparison['difference_(MIMIC_minus_India)'] = (
    percent_comparison['MIMIC OBS'] - percent_comparison['India InternVL']
).round(2)
display(percent_comparison)

mimic_top = summary_df[summary_df['dataset'] == 'MIMIC OBS'].sort_values('count', ascending=False).iloc[0]
india_top = summary_df[summary_df['dataset'] == 'India InternVL'].sort_values('count', ascending=False).iloc[0]

print(f"MIMIC OBS is dominated by '{mimic_top['label']}' ({mimic_top['count']} rows, {mimic_top['percent']:.2f}%).")
print(f"India InternVL is dominated by '{india_top['label']}' ({india_top['count']} rows, {india_top['percent']:.2f}%).")
print(
    'The unrelated share is ' 
    f"{percent_table.loc['unrelated', 'MIMIC OBS']:.2f}% in MIMIC OBS and "
    f"{percent_table.loc['unrelated', 'India InternVL']:.2f}% in India InternVL."
)